In [ ]:
# initialize model

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()

model = ChatOpenAI(model="gpt-5.2")

c:\Users\Yaseen T P\Desktop\Gordian Redis system\gorenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# define pydantic model for output

from pydantic import BaseModel
from typing import Optional, Literal, List


class Step2Output(BaseModel):
    status: Literal["ok", "fail"]

    expression: Optional[str]
    """
    Mathematical expression using variable `price`,
    e.g. '{price} * (1000 / 3.85)'
    """

    explanation: Optional[str]
    """
    Short factual justification describing which explicit evidence
    was used to form the expression.
    Must NOT include calculations or algebraic transformations.
    """

    evidence_used: List[str]
    """
    Exact seller-visible texts or table rows used
    (e.g. 'Weight/Metre (Kg): 3.85')
    """


In [ ]:
# create structured ouptut model

structured_model = model.with_structured_output(Step2Output)

In [ ]:
# Support to verify the generated expression is correct and safe to evaluate

import ast
import operator as op


# Allowed operators
_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
}


class MathExpressionError(Exception):
    pass


def evaluate_math_expression(expression: str) -> float:
    """
    Safely evaluate a mathematical expression consisting of
    numbers, + - * / and parentheses.

    Args:
        expression (str): math expression, e.g. "(1000 / 3.85)"

    Returns:
        float: evaluated result

    Raises:
        MathExpressionError: if expression is invalid or unsafe
    """

    try:
        parsed = ast.parse(expression, mode="eval")
        return _eval_node(parsed.body)

    except Exception as e:
        raise MathExpressionError(f"Invalid math expression: {expression}") from e


def _eval_node(node):
    if isinstance(node, ast.Num):  # Python <3.8
        return node.n

    if isinstance(node, ast.Constant):  # Python 3.8+
        if isinstance(node.value, (int, float)):
            return node.value
        raise MathExpressionError("Only numeric constants allowed")

    if isinstance(node, ast.BinOp):
        if type(node.op) not in _ALLOWED_OPERATORS:
            raise MathExpressionError(f"Operator {type(node.op)} not allowed")

        left = _eval_node(node.left)
        right = _eval_node(node.right)

        return _ALLOWED_OPERATORS[type(node.op)](left, right)

    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op, ast.USub):
            return -_eval_node(node.operand)
        raise MathExpressionError("Unary operator not allowed")

    raise MathExpressionError(f"Unsupported expression element: {type(node)}")


import re
from typing import Dict


def execute_llm_expression(
    raw_expression: str,
    variables: Dict[str, float],
) -> float:
    """
    Execute a math expression produced by the LLM.

    Steps:
    1. Template variables (price -> {price})
    2. Substitute runtime values
    3. Validate expression safety
    4. Evaluate using safe math executor

    Args:
        raw_expression: e.g. "price * (1000 / 0.89)"
        variables: e.g. {"price": 10}

    Returns:
        float result

    Raises:
        ValueError if expression is unsafe or invalid
    """

    # --- Step 1: template variables ---
    for var in variables.keys():
        raw_expression = re.sub(
            rf"\b{var}\b",
            f"{{{var}}}",
            raw_expression
        )

    # --- Step 2: substitute variables ---
    expr = raw_expression
    for key, value in variables.items():
        if not isinstance(value, (int, float)):
            raise ValueError(f"Invalid value for {key}: {value}")
        expr = expr.replace(f"{{{key}}}", str(value))

    # --- Step 3: ensure no unresolved placeholders remain ---
    if re.search(r"\{[a-zA-Z_]+\}", expr):
        raise ValueError(f"Incorrect Math Exp: {expr}")

    # --- Step 4: validate allowed characters only ---
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expr):
        raise ValueError(f"Incorrect Math Exp: {expr}")

    # --- Step 5: evaluate using Step-3 calculator ---
    return evaluate_math_expression(expr)



PROMPTS

In [ ]:
# Version2 :  two systems prompts;switch to prompt that emphasize "Comments" if information cannot be inferred from webpage
  
SYSTEM_PROMPT_SELLER = """You are a pricing conversion engine.

Your task is to generate a MATHEMATICAL EXPRESSION to convert given unit price to target unit price
from the priced quantity unit to the target unit.

You are provided with:
- a screenshot of webpage of a product
- "target_unit"
- a user "comment" might be helpful to derive a MATHEMATICAL EXPRESSION

<INSTRUCTIONS>
- Infer the quanity and given unit of the product from the webpage
- If the given unit and target unit is not same, identify the conversion factor from webpage. For example the qiven quantity is in liter and target quantity in Kg, check for denisty is the specs or more info table
- If the given unit and target unit is same, mathematical experssion is straight forward


STRICT RULES:
- Do NOT assume material properties or typical values.
- Do NOT restate steps.
- Do NOT calculate numeric results.
- Do NOT algebraically rewrite or invert expressions.
- Use ONLY numeric values explicitly visible in the screenshot or evidence.
- Do NOT assume material properties.
- Do NOT invent constants.
- If the required conversion factor is not explicitly visible, return FAIL.

VARIABLE RULES:
- Use the variable name `price` for the price that will be substituted
- All other values must be numeric literals taken directly from the screenshot.

ALLOWED OPERATIONS:
- +  -  *  /
- parentheses ( )


- return a valid math expression with variable "price" can be subtituted in python math evaulator (e.g. `{/{price}/} * (1000 / 3.85)`), OR
- the single word: FAIL

"""


SYSTEM_PROMPT_COMMENT = """You are a pricing conversion engine operating in COMMENT-PRIMARY mode.

The webpage does NOT provide an explicit conversion between the priced unit
and the target unit.

Your task is to generate a MATHEMATICAL EXPRESSION using the user-provided
comment as the PRIMARY source of quantity or conversion information.

You are provided with:
- target_unit
- a user comment

RULES:
- Treat the comment as a claim about how the product is sold.
- Validate that the comment does NOT contradict visible product details
  (dimensions, material type, product category).
- Do NOT assume typical sizes or industry standards.
- If the comment is ambiguous or contradicts product details, return FAIL.
- You may use simple arithmetic implied directly by the comment
  (e.g. "price is for 5m" → divide by 5).

STRICT CONSTRAINTS:
- Do NOT invent numeric values not present in the comment.
- Do NOT calculate final numeric results.
- Do NOT rewrite or invert expressions.
- Use variable `price` only.

OUTPUT:
Return a JSON object matching the schema.
If conversion cannot be safely derived, return FAIL.

Billing accuracy is required."""




In [ ]:
# set logger
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger("__info__")


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
import asyncio
import base64
import pandas as pd


async def run_with_retry(
    *,
    structured_model,
    encoded_image: str,
    row,
    max_retries: int = 1,
):
    """
    Try seller-evidence prompt first.
    Retry once with comment-primary prompt if needed.
    """

    attempts = [
        ("SCREENSHOT", SYSTEM_PROMPT_SELLER),
        ("COMMENT", SYSTEM_PROMPT_COMMENT),
    ]

    last_error = None

    for attempt_name, system_prompt in attempts[: max_retries + 1]:
        try:
            user_prompt = (
                f"Comment: {row.get('Comments')}\n"
                f"Target Unit: {row.get('TargetUnit')}\n"
                f"Screenshot:"
            )

            human_msg = HumanMessage(
                [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "image",
                        "base64": encoded_image,
                        "mime_type": "image/png",
                        "detail": "low",
                    },
                ]
            )

            system_msg = SystemMessage(system_prompt)

            response: Step2Output = await structured_model.ainvoke(
                [system_msg, human_msg]
            )

            response_json = response.model_dump()

            logger.info(
                            "LLM_RESPONSE | mode=%s | response=%s | comment=%s",
                            attempt_name,
                            response_json['explanation'],
                            row["Comments"],
                            )

            if response.status == "ok" and response.expression:
                logger.info(
                    "STEP2_SUCCESS | mode=%s | expr=%s",
                    attempt_name,
                    response.expression,
                )
                return response, attempt_name, None

            last_error = f"{attempt_name}_INVALID_OUTPUT"
            

        except Exception as e:
            last_error = f"{attempt_name}_ERROR: {e}"
            logger.error(last_error)

    return response, attempt_name, last_error


async def run_and_validate(
    dataset: pd.DataFrame,
    structured_model,
    semaphore_limit: int = 5,
) -> pd.DataFrame:

    semaphore = asyncio.Semaphore(semaphore_limit)

    dataset = dataset.copy()
    dataset["final_unit_price"] = None
    dataset["final_expression"] = None
    dataset["explanation"] = None
    dataset["error"] = None
    dataset["conversion_mode"] = None

    async def process_row(idx: int, row: pd.Series):
        async with semaphore:
            try:
                extracted_price = float(row["ExtractedPrice"])
                image_path = row["img_path"]

                # ---- encode image ----
                with open(image_path, "rb") as f:
                    encoded_image = base64.b64encode(f.read()).decode("utf-8")

                # ---- Step 2 (with retry) ----
                response, mode, error = await run_with_retry(
                    structured_model=structured_model,
                    encoded_image=encoded_image,
                    row=row,
                )

                

                if error:
                    dataset.at[idx, "error"] = error
        

                expr = response.expression
                explanation = response.explanation

                response_json = response.model_dump()

                logger.info(
                            "LLM_RESPONSE | row=%s | mode=%s | response=%s | comment=%s",
                            idx,
                            mode,
                            response_json['explanation'],
                            row["Comments"],
                            )

                # ---- Step 3 (execution) ----
                try:
                    result = execute_llm_expression(
                        raw_expression=expr,
                        variables={"price": extracted_price},
                    )
                except Exception as exec_error:
                    dataset.at[idx, "error"] = f"EXECUTION_ERROR: {exec_error}"
                    dataset.at[idx, "final_expression"] = expr
                    dataset.at[idx, "explanation"] = explanation
                    return

    
                dataset.at[idx, "final_unit_price"] = result
                dataset.at[idx, "final_expression"] = expr
                dataset.at[idx, "explanation"] = explanation
                dataset.at[idx, "conversion_mode"] = mode

                logger.info(
                    "ROW_SUCCESS | row=%s | mode=%s | result=%s",
                    idx,
                    mode,
                    result,
                )

            except Exception as fatal_error:
                logger.exception("FATAL_ERROR | row=%s", idx)
                dataset.at[idx, "error"] = f"FATAL_ERROR: {fatal_error}"

    tasks = [
        process_row(idx, row)
        for idx, row in dataset.iterrows()
    ]

    await asyncio.gather(*tasks)

    return dataset




In [ ]:
# load dataset for testing
import pandas as pd
source = 'metals4u.co'
dataset = pd.read_excel(f'./dataset/{source}_with_img_path.xlsx')
dataset = dataset.rename(columns={'Unit\n':'TargetUnit','Rate £':'ExtractedPrice'})


In [ ]:
output = await run_and_validate(
    dataset=dataset[:5],
    structured_model=structured_model,
    semaphore_limit=5,
)

2026-05-04 17:04:44,338 | INFO | Retrying request to /chat/completions in 0.440734 seconds
2026-05-04 17:04:44,339 | INFO | Retrying request to /chat/completions in 0.491866 seconds
2026-05-04 17:04:44,340 | INFO | Retrying request to /chat/completions in 0.428499 seconds
2026-05-04 17:04:44,341 | INFO | Retrying request to /chat/completions in 0.493165 seconds
2026-05-04 17:04:44,345 | INFO | Retrying request to /chat/completions in 0.497186 seconds
2026-05-04 17:04:44,790 | INFO | Retrying request to /chat/completions in 0.831807 seconds
2026-05-04 17:04:44,791 | INFO | Retrying request to /chat/completions in 0.760135 seconds
2026-05-04 17:04:44,869 | INFO | Retrying request to /chat/completions in 0.879015 seconds
2026-05-04 17:04:44,871 | INFO | Retrying request to /chat/completions in 0.750246 seconds
2026-05-04 17:04:44,871 | INFO | Retrying request to /chat/completions in 0.784222 seconds
2026-05-04 17:04:45,549 | ERROR | SCREENSHOT_ERROR: Connection error.
2026-05-04 17:04:45,

In [21]:
output

,Unnamed: 0,Code,Code.1,Description,Type,TargetUnit,ExtractedPrice,Extracted Price,New Rate (ex VAT),New Rate INCL VAT),...,Web,Status,Unnamed: 21,InApp Comment,img_path,final_unit_price,final_expression,explanation,error,conversion_mode
0,NaN,BME4001,BME4001,High Yield bar reinforcement (Grade 500) 25mm,1.0,t,3923.40,NaN,NaN,NaN,...,Yes,NaN,NaN,NaN,screenshots\metals4u.co\https_www.metals4u.co....,None,None,None,FATAL_ERROR: cannot access local variable 'res...,None
1,NaN,BME4002,BME4002,High Yield bar reinforcement (Grade 500) 20mm,1.0,t,3178.98,NaN,NaN,NaN,...,Yes,NaN,NaN,NaN,screenshots\metals4u.co\https_www.metals4u.co....,None,None,None,FATAL_ERROR: cannot access local variable 'res...,None
2,NaN,BME4003,BME4003,High Yield bar reinforcement (Grade 500) 16mm,1.0,t,3138.30,NaN,NaN,NaN,...,Yes,NaN,NaN,NaN,screenshots\metals4u.co\https_www.metals4u.co....,None,None,None,FATAL_ERROR: cannot access local variable 'res...,None
3,NaN,BME4004,BME4004,High Yield bar reinforcement (Grade 500) 12mm,1.0,t,3558.16,NaN,NaN,NaN,...,Yes,NaN,NaN,NaN,screenshots\metals4u.co\https_www.metals4u.co....,None,None,None,FATAL_ERROR: cannot access local variable 'res...,None
4,NaN,BME4005,BME4005,High Yield bar reinforcement (Grade 500) 10mm,1.0,t,4043.76,NaN,NaN,NaN,...,Yes,NaN,NaN,NaN,screenshots\metals4u.co\https_www.metals4u.co....,None,None,None,FATAL_ERROR: cannot access local variable 'res...,None


In [22]:
# save results
output.to_excel(f"./result/{source}_output.xlsx")